In [51]:
import cv2
import numpy as np
import pandas as pd

# Basic Functions

## Entropy Calculation

In [42]:
def my_entropy(input_image):

    arr = input_image.flatten().astype(np.int64)

    if arr.min() != 1:
        arr = arr - arr.min() + 1

    p = np.zeros(arr.max(), dtype=np.float64)
    for v in arr:
        p[v - 1] += 1

    p = p / p.sum()
    p = p[p != 0]
    entropy = np.sum(-p * np.log2(p))

    return entropy

## PSNR Calculation

In [43]:
def next_power_of_two(x):
    return 1 << (x - 1).bit_length()

def peak_signal_noise_ratio(image1: np.ndarray, image2: np.ndarray):
    if image1.shape != image2.shape:
        return -1

    max_intensity = max(image1.max(), image2.max())

    next_pow2 = next_power_of_two(int(max_intensity + 1))
    max_value = next_pow2 - 1
    max_value_sq = max_value ** 2

    mse = np.mean((image1.astype(np.float64) - image2.astype(np.float64)) ** 2)

    if mse == 0:
        return np.inf

    return 10 * np.log10(max_value_sq / mse)

# Stereo Compression

## Helper functions

### Image Padding

In [44]:
def pad_image(img, block_size=16):
    H, W = img.shape
    pad_h = (block_size - H % block_size) % block_size
    pad_w = (block_size - W % block_size) % block_size
    return np.pad(img, ((0, pad_h), (0, pad_w)), mode='edge'), H, W

### 3 Step Search (3SS)

In [45]:
def three_step_search(reference, curr_block, y, x, block_size, search_range):
    H, W = reference.shape

    step = 2 ** int(np.floor(np.log2(search_range)))
    best_mv = (0, 0)

    while step >= 1:
        best_sad = np.inf
        cy, cx = best_mv

        for dy in [-step, 0, step]:
            for dx in [-step, 0, step]:
                ny = cy + dy
                nx = cx + dx

                ref_y = y + ny
                ref_x = x + nx

                if (ref_y < 0 or ref_x < 0 or
                        ref_y + block_size > H or
                        ref_x + block_size > W):
                    continue

                ref_block = reference[ref_y:ref_y+block_size, ref_x:ref_x+block_size]
                sad = np.sum(np.abs(curr_block.astype(np.int16) - ref_block.astype(np.int16)))

                if sad < best_sad:
                    best_sad = sad
                    best_mv = (ny, nx)

        step //= 2

    return best_mv

### RGB2RCT & RCT2RGB

In [46]:
def rgb_to_rct(rgb):
    rgb = rgb.astype(np.int16)

    R = rgb[:, :, 0]
    G = rgb[:, :, 1]
    B = rgb[:, :, 2]

    Y  = (R + 2*G + B) >> 2
    Cb = B - G
    Cr = R - G

    return np.stack([Y, Cb, Cr], axis=2)

def rct_to_rgb(rct):
    Y  = rct[:, :, 0]
    Cb = rct[:, :, 1]
    Cr = rct[:, :, 2]

    G = Y - ((Cb + Cr) >> 2)
    R = Cr + G
    B = Cb + G

    rgb = np.stack([R, G, B], axis=2)
    return rgb.astype(np.uint8)

### MED Predictor

In [55]:
def med_predictor(input_image):
    input_image = input_image.astype(np.int16)
    H, W = input_image.shape

    error_image = np.zeros((H, W), dtype=np.int16)

    padded_Image = np.pad(input_image, ((1, 0), (1, 0)), mode='constant', constant_values=0)

    for i in range(1, H + 1):
        for j in range(1, W + 1):

            a = padded_Image[i, j - 1]
            b = padded_Image[i - 1, j]
            c = padded_Image[i - 1, j - 1]

            if c >= max(a, b):
                x = min(a, b)
            elif c <= min(a, b):
                x = max(a, b)
            else:
                x = a + b - c

            error_image[i - 1, j - 1] = input_image[i - 1, j - 1] - x

    return error_image


def med_reconstructor(error_image):
    error_image = error_image.astype(np.int16)
    H, W = error_image.shape

    prediction = np.zeros((H + 1, W + 1), dtype=np.int16)
    reconstructed_image = np.zeros((H, W), dtype=np.uint8)

    for i in range(1, H + 1):
        for j in range(1, W + 1):
            a = prediction[i, j - 1]
            b = prediction[i - 1, j]
            c = prediction[i - 1, j - 1]

            if c >= max(a, b):
                x = min(a, b)
            elif c <= min(a, b):
                x = max(a, b)
            else:
                x = a + b - c

            prediction[i, j] = x + error_image[i - 1, j - 1]
            reconstructed_image[i - 1, j - 1] = np.uint8(prediction[i, j])

    return reconstructed_image

## Encoder

In [47]:
def encoder(left_rgb, right_rgb, block_size=16, search_range=22):

    # 1-RGB to RCT
    left_rgb  = left_rgb.astype(np.int16)
    right_rgb = right_rgb.astype(np.int16)

    R = left_rgb[:, :, 0]
    G = left_rgb[:, :, 1]
    B = left_rgb[:, :, 2]
    left_Y  = (R + 2*G + B) >> 2
    left_Cb = B - G
    left_Cr = R - G


    R = right_rgb[:, :, 0]
    G = right_rgb[:, :, 1]
    B = right_rgb[:, :, 2]
    right_Y  = (R + 2*G + B) >> 2
    right_Cb = B - G
    right_Cr = R - G

    # 2-Padding
    left_Y, H, W = pad_image(left_Y, block_size)
    left_Cb, _, _  = pad_image(left_Cb, block_size)
    left_Cr, _, _  = pad_image(left_Cr, block_size)

    right_Y, _, _ = pad_image(right_Y, block_size)
    right_Cb, _, _ = pad_image(right_Cb, block_size)
    right_Cr, _, _ = pad_image(right_Cr, block_size)


    # 3-Motion Estimation on Y
    H_pad, W_pad = right_Y.shape
    residual_Y = np.zeros_like(right_Y, dtype=np.int16)

    mv_h = H_pad // block_size
    mv_w = W_pad // block_size
    motion_vectors = np.zeros((mv_h, mv_w, 2), dtype=np.int16)

    for by in range(mv_h):
        for bx in range(mv_w):
            y = by * block_size
            x = bx * block_size

            curr_block = right_Y[y:y+block_size, x:x+block_size]

            mv = three_step_search(left_Y, curr_block, y, x, block_size, search_range)
            motion_vectors[by, bx] = mv

            ref_y = y + mv[0]
            ref_x = x + mv[1]

            pred = left_Y[ref_y:ref_y+block_size, ref_x:ref_x+block_size]
            residual_Y[y:y+block_size, x:x+block_size] = (curr_block.astype(np.int16) - pred.astype(np.int16))


    # 4-Chroma residuals (reuse MV)
    residual_Cb = np.zeros_like(right_Cb, dtype=np.int16)
    residual_Cr = np.zeros_like(right_Cr, dtype=np.int16)

    for by in range(mv_h):
        for bx in range(mv_w):
            y = by * block_size
            x = bx * block_size

            dy, dx = motion_vectors[by, bx]

            pred_cb = left_Cb[y+dy:y+dy+block_size, x+dx:x+dx+block_size]
            pred_cr = left_Cr[y+dy:y+dy+block_size, x+dx:x+dx+block_size]

            residual_Cb[y:y+block_size, x:x+block_size] = (right_Cb[y:y+block_size, x:x+block_size] - pred_cb)
            residual_Cr[y:y+block_size, x:x+block_size] = (right_Cr[y:y+block_size, x:x+block_size] - pred_cr)

    return motion_vectors, residual_Y, residual_Cb, residual_Cr, (H, W)

## Decoder

In [48]:
def decoder(left_rgb, motion_vectors, residual_Y, residual_Cb, residual_Cr, orig_shape, block_size=16):
    # 1-RCT of left
    left_rgb = left_rgb.astype(np.int16)

    R = left_rgb[:, :, 0]
    G = left_rgb[:, :, 1]
    B = left_rgb[:, :, 2]

    left_Y  = (R + 2*G + B) >> 2
    left_Cb = B - G
    left_Cr = R - G

    # 2-Padding
    left_Y, _, _  = pad_image(left_Y, block_size)
    left_Cb, _, _ = pad_image(left_Cb, block_size)
    left_Cr, _, _ = pad_image(left_Cr, block_size)

    H_pad, W_pad = residual_Y.shape
    mv_h, mv_w, _ = motion_vectors.shape

    # 3-Reconstruct
    recon_Y = np.zeros_like(left_Y, dtype=np.int16)
    for by in range(mv_h):
        for bx in range(mv_w):
            y = by * block_size
            x = bx * block_size
            dy, dx = motion_vectors[by, bx]
            pred = left_Y[y+dy:y+dy+block_size, x+dx:x+dx+block_size]
            recon_Y[y:y+block_size, x:x+block_size] = (pred + residual_Y[y:y+block_size, x:x+block_size])

    recon_Cb = np.zeros_like(left_Cb, dtype=np.int16)
    for by in range(mv_h):
        for bx in range(mv_w):
            y = by * block_size
            x = bx * block_size
            dy, dx = motion_vectors[by, bx]
            pred = left_Cb[y+dy:y+dy+block_size, x+dx:x+dx+block_size]
            recon_Cb[y:y+block_size, x:x+block_size] = (pred + residual_Cb[y:y+block_size, x:x+block_size])


    recon_Cr = np.zeros_like(left_Cr, dtype=np.int16)
    for by in range(mv_h):
        for bx in range(mv_w):
            y = by * block_size
            x = bx * block_size
            dy, dx = motion_vectors[by, bx]
            pred = left_Cr[y+dy:y+dy+block_size, x+dx:x+dx+block_size]
            recon_Cr[y:y+block_size, x:x+block_size] = (pred + residual_Cr[y:y+block_size, x:x+block_size])

    # 5-Crop to original shape
    H, W = orig_shape
    recon_Y  = recon_Y[:H, :W]
    recon_Cb = recon_Cb[:H, :W]
    recon_Cr = recon_Cr[:H, :W]

    # 8-Inverse RCT (Rct to RGB)
    G = recon_Y - ((recon_Cb + recon_Cr) >> 2)
    R = recon_Cr + G
    B = recon_Cb + G

    recon_rgb = np.stack([R, G, B], axis=2)
    return recon_rgb

In [54]:
result1 = []

test_images = ['Alovera', 'Books', 'Bowling', 'Chess', 'Dolls', 'Snowman', 'Teddy']

for addr in test_images:
    left_image = cv2.imread(f'Stereo_Pairs/{addr}/Image_1.png', cv2.IMREAD_UNCHANGED)
    right_image = cv2.imread(f'Stereo_Pairs/{addr}/Image_2.png', cv2.IMREAD_UNCHANGED)

    mv, error_Y, error_Cb, error_Cr, (H, W) = encoder(left_rgb=left_image, right_rgb=right_image, block_size=16)
    recon = decoder(left_rgb=left_image, motion_vectors=mv,residual_Y=error_Y, residual_Cb=error_Cb, residual_Cr=error_Cr,
                    orig_shape=(H, W), block_size=16)

    psnr = peak_signal_noise_ratio(image1=right_image, image2=recon)

    mv_entropy = None

    error_Y_entropy = my_entropy(error_Y)
    error_Cb_entropy = my_entropy(error_Cb)
    error_Cr_entropy = my_entropy(error_Cr)

    left_image_entropy = my_entropy(left_image)

    result1.append({'Image Name': f"{addr}.png", "Y_error_entropy": error_Y_entropy, "Cb_error_entropy": error_Cb_entropy,
                    "Cr_error_entropy": error_Cr_entropy, "left_image_entropy": left_image_entropy, 'Reconstruction_right_psnr': psnr})


result1 = pd.DataFrame(result1)
# avg_row = { "Image Name": "Average",
#             "Red Channel": result1["Red Channel"].mean(),
#             "Green Channel": result1["Green Channel"].mean(),
#             "Blue Channel": result1["Blue Channel"].mean(),
#             "Average": result1["Average"].mean()}
#
# result1 = pd.concat([result1, pd.DataFrame([avg_row])], ignore_index=True)
result1

,Image Name,Y_error_entropy,Cb_error_entropy,Cr_error_entropy,left_image_entropy,Reconstruction_right_psnr
0,Alovera.png,5.877904,5.564767,4.990819,7.483377,inf
1,Books.png,6.624343,5.280001,5.439132,7.443833,inf
2,Bowling.png,6.753776,5.415654,5.621178,6.576005,-1.0
3,Chess.png,6.510107,5.267062,5.141032,7.515921,inf
4,Dolls.png,6.595397,6.888201,6.304179,7.753873,inf
5,Snowman.png,6.697721,5.321298,6.147420,7.709072,inf
6,Teddy.png,5.691783,6.086000,6.298507,7.794844,inf
